In [59]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

In [60]:
import our_steady_state_model as ss

In [61]:
df = pd.read_csv("Weather and Temperature Logs(February 2026).csv")
df.head()

,Time Stamp,Current Weather,Temp (C),Humidity,Barometric Pressue (Pa),Wind Direction,Wind Speed (m/s),Wind Chill (C),Heat Index (C)
0,2/1/2026 0:15:00,Cloudy,-11.4,60.644017,101420.0,320.0,14.76,-18.385102,NaN
1,2/1/2026 0:35:00,Mostly Cloudy,-11.5,61.647019,101390.0,310.0,9.36,-16.787109,NaN
2,2/1/2026 0:55:00,Clear,-11.6,61.620911,101320.0,310.0,22.32,-20.310614,NaN
3,2/1/2026 1:15:00,Clear,-12.1,64.140316,101360.0,320.0,16.56,-19.708570,NaN
4,2/1/2026 1:35:00,Clear,-12.3,64.090342,101390.0,310.0,12.96,-18.978616,NaN


In [62]:
# Wind direction is probably not needed, heat index column is mostly empty
df = df.drop("Wind Direction", axis=1)
df = df.drop("Heat Index (C)", axis=1)

In [63]:
df.head()

,Time Stamp,Current Weather,Temp (C),Humidity,Barometric Pressue (Pa),Wind Speed (m/s),Wind Chill (C)
0,2/1/2026 0:15:00,Cloudy,-11.4,60.644017,101420.0,14.76,-18.385102
1,2/1/2026 0:35:00,Mostly Cloudy,-11.5,61.647019,101390.0,9.36,-16.787109
2,2/1/2026 0:55:00,Clear,-11.6,61.620911,101320.0,22.32,-20.310614
3,2/1/2026 1:15:00,Clear,-12.1,64.140316,101360.0,16.56,-19.708570
4,2/1/2026 1:35:00,Clear,-12.3,64.090342,101390.0,12.96,-18.978616


In [64]:
# Our reference values
R_wall = 2.3
A_envelope = 70
ACH = 0.5
volume = 30

In [65]:
df["R_wall"] = R_wall
df["A_envelope"] = A_envelope
df["ACH"] = ACH
df["volume"] = volume

In [66]:
df.head()

,Time Stamp,Current Weather,Temp (C),Humidity,Barometric Pressue (Pa),Wind Speed (m/s),Wind Chill (C),R_wall,A_envelope,ACH,volume
0,2/1/2026 0:15:00,Cloudy,-11.4,60.644017,101420.0,14.76,-18.385102,2.3,70,0.5,30
1,2/1/2026 0:35:00,Mostly Cloudy,-11.5,61.647019,101390.0,9.36,-16.787109,2.3,70,0.5,30
2,2/1/2026 0:55:00,Clear,-11.6,61.620911,101320.0,22.32,-20.310614,2.3,70,0.5,30
3,2/1/2026 1:15:00,Clear,-12.1,64.140316,101360.0,16.56,-19.708570,2.3,70,0.5,30
4,2/1/2026 1:35:00,Clear,-12.3,64.090342,101390.0,12.96,-18.978616,2.3,70,0.5,30


In [67]:
# Conversions
df["Temp (C)"] = ss.c_to_f(df["Temp (C)"])
df["Wind Chill (C)"] = df["Wind Chill (C)"]*(9/5)
df["Humidity"] = df["Humidity"] / 100
df = df.rename(columns={"Temp (C)":"Temp (F)", "Wind Chill (C)":"Wind Chill (F)"})

In [68]:
df.head()

,Time Stamp,Current Weather,Temp (F),Humidity,Barometric Pressue (Pa),Wind Speed (m/s),Wind Chill (F),R_wall,A_envelope,ACH,volume
0,2/1/2026 0:15:00,Cloudy,11.48,0.606440,101420.0,14.76,-33.093184,2.3,70,0.5,30
1,2/1/2026 0:35:00,Mostly Cloudy,11.30,0.616470,101390.0,9.36,-30.216796,2.3,70,0.5,30
2,2/1/2026 0:55:00,Clear,11.12,0.616209,101320.0,22.32,-36.559105,2.3,70,0.5,30
3,2/1/2026 1:15:00,Clear,10.22,0.641403,101360.0,16.56,-35.475425,2.3,70,0.5,30
4,2/1/2026 1:35:00,Clear,9.86,0.640903,101390.0,12.96,-34.161509,2.3,70,0.5,30


In [69]:
# For now assume set temp is 72 F
df["User Heat Loss"] = None
df["Verdict"] = None
df["Internal Humidity"] = None
df["Building Heat Loss"] = None
df["Wall Temp (F)"] = None
for i in range(len(df)):
    user_q_total, verdict, RH_internal, Q_total, T_wall_F = ss.steady_state_model(72, df.at[i, "Temp (F)"], df.at[i, "Humidity"], print_output=False)
    df.at[i, "User Heat Loss"] = user_q_total
    df.at[i, "Verdict"] = verdict
    df.at[i, "Internal Humidity"] = RH_internal
    df.at[i, "Building Heat Loss"] = Q_total
    df.at[i, "Wall Temp (F)"] = T_wall_F


In [70]:
df.head()

,Time Stamp,Current Weather,Temp (F),Humidity,Barometric Pressue (Pa),Wind Speed (m/s),Wind Chill (F),R_wall,A_envelope,ACH,volume,User Heat Loss,Verdict,Internal Humidity,Building Heat Loss,Wall Temp (F)
0,2/1/2026 0:15:00,Cloudy,11.48,0.606440,101420.0,14.76,-33.093184,2.3,70,0.5,30,62.440033,Good,0.05796,1192.236691,68.014523
1,2/1/2026 0:35:00,Mostly Cloudy,11.30,0.616470,101390.0,9.36,-30.216796,2.3,70,0.5,30,62.456026,Good,0.058449,1195.782669,68.002669
2,2/1/2026 0:55:00,Clear,11.12,0.616209,101320.0,22.32,-36.559105,2.3,70,0.5,30,62.476918,Good,0.057958,1199.328647,67.990816
3,2/1/2026 1:15:00,Clear,10.22,0.641403,101360.0,16.56,-35.475425,2.3,70,0.5,30,62.569111,Good,0.057954,1217.058539,67.931547
4,2/1/2026 1:35:00,Clear,9.86,0.640903,101390.0,12.96,-34.161509,2.3,70,0.5,30,62.610823,Good,0.056983,1224.150495,67.90784


Not sure if we will use wind speed, wind chill, or barometric pressure.